### 3.4. Datasets merge
After cleaning all three main datasets, it is now time to merge all data into the same table to work on our exploratory data analysis and especially looking for insights.  
But before merging, it is best practice to make a final check of harmonization data.

#### Final checks

In [2]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define clean dataset folder path
clean_dataset_path = Path('../dataset/clean')

# access cleaned datasets
prices_data = pd.read_csv(clean_dataset_path / 'pun-index-clean.csv')
gen_data = pd.read_csv(clean_dataset_path / 'gen-clean.csv')
load_data = pd.read_csv(clean_dataset_path / 'load-italy-clean.csv')

In [4]:
prices_data

,datetime,Date,Prices
0,2021-01-01 00:00:00,2021-01-01,50.87000
1,2021-01-01 01:00:00,2021-01-01,48.19000
2,2021-01-01 02:00:00,2021-01-01,44.67966
3,2021-01-01 03:00:00,2021-01-01,42.92193
4,2021-01-01 04:00:00,2021-01-01,40.39151
...,...,...,...
43819,2025-12-31 19:00:00,2025-12-31,121.00000
43820,2025-12-31 20:00:00,2025-12-31,113.96499
43821,2025-12-31 21:00:00,2025-12-31,111.20000
43822,2025-12-31 22:00:00,2025-12-31,113.56500


In [5]:
gen_data

,Date,Geothermal,Hydro,Photovoltaic,Self-consumption,Thermal,Wind,TOT Generation,Renewables weight
0,2021-01-01 00:00:00,0.62,3.43,0.0,2.058,14.55,2.60,21.20,0.313679
1,2021-01-01 01:00:00,0.62,3.03,0.0,1.952,13.68,2.75,20.08,0.318725
2,2021-01-01 02:00:00,0.62,2.79,0.0,1.812,12.77,2.63,18.81,0.321106
3,2021-01-01 03:00:00,0.62,2.60,0.0,1.735,12.12,2.62,17.96,0.325167
4,2021-01-01 04:00:00,0.62,2.60,0.0,1.733,12.06,2.61,17.89,0.325880
...,...,...,...,...,...,...,...,...,...
43814,2025-12-31 19:00:00,2.40,15.00,0.0,13.676,90.60,9.32,117.32,0.227753
43815,2025-12-31 20:00:00,2.40,11.28,0.0,12.580,85.32,8.32,107.32,0.204994
43816,2025-12-31 21:00:00,2.40,8.44,0.0,10.764,80.44,6.68,97.96,0.178849
43817,2025-12-31 22:00:00,2.40,7.12,0.0,9.708,74.60,5.44,89.56,0.167039


In [6]:
load_data

,Date,Total Load,Load forecasted
0,2021-01-01 00:00:00,24640.25075,24614.24975
1,2021-01-01 01:00:00,22909.00000,23971.99975
2,2021-01-01 02:00:00,21044.49975,22554.74950
3,2021-01-01 03:00:00,19907.99975,21153.49975
4,2021-01-01 04:00:00,19578.49950,20336.75000
...,...,...,...
43819,2025-12-31 19:00:00,37002.25000,36421.74925
43820,2025-12-31 20:00:00,34212.50000,32732.99975
43821,2025-12-31 21:00:00,31110.24950,29597.99925
43822,2025-12-31 22:00:00,29077.25000,27451.50025


It is noticeable that prices_data dataset has Date column which is different from the other dataset of which we want to merge the table.  
In this some adjustments are required.

In [7]:
# remove old Date column
prices_data = prices_data.drop(columns='Date')

# rename datetime column into Date
prices_data = prices_data.rename(columns={
    'datetime': 'Date'
})

prices_data

,Date,Prices
0,2021-01-01 00:00:00,50.87000
1,2021-01-01 01:00:00,48.19000
2,2021-01-01 02:00:00,44.67966
3,2021-01-01 03:00:00,42.92193
4,2021-01-01 04:00:00,40.39151
...,...,...
43819,2025-12-31 19:00:00,121.00000
43820,2025-12-31 20:00:00,113.96499
43821,2025-12-31 21:00:00,111.20000
43822,2025-12-31 22:00:00,113.56500


In [9]:
prices_data.describe()

,Prices
count,43824.000000
mean,156.197214
std,104.706032
min,0.000000
25%,98.520517
50%,121.000000
75%,179.341460
max,870.000000


Now that the column uniformation is adjusted, we have to look for duplicates.

In [12]:
print(prices_data['Date'].duplicated().sum())
print(gen_data['Date'].duplicated().sum())
print(load_data['Date'].duplicated().sum())

5
0
0


Since prices_data has duplicates, probably because in October in Italy there is Solar time, which basically is adding one hour once a year, for economy and power saving reasons, we ha ve to correct this inconsistency.  
The approach selected is deleting duplicate rows, which are 5 out of 43824 (0.001% of deleted data).

In [ ]:
prices_data = prices_data.drop_duplicates(
    subset='Date', 
    keep='first'
)

In [15]:
prices_data.describe()

,Prices
count,43819.000000
mean,156.200777
std,104.710843
min,0.000000
25%,98.520000
50%,121.000000
75%,179.342610
max,870.000000


#### Progressive merge
Now that tables are normalized, it is time to progressively merge the dataset.

In [16]:
# merge prices_data and gen_data  
merged_data = prices_data.merge(
    gen_data,
    on='Date', 
    how='left'
)

# merge prices_data, gen_data and load_data in merged_data
merged_data = merged_data.merge(
    load_data,
    on='Date', 
    how='left'
)

In [ ]:
# convert Date to datetime object again (the merge modified its Type)
merged_data['Date'] = pd.to_datetime(merged_data['Date'])

In [ ]:
# sort Date just in case
merged_data = merged_data.sort_values('Date')

In [ ]:
# find NaN rows
merged_data.isna().sum().sort_values(ascending=False)


Hydro                5
Geothermal           5
Self-consumption     5
Photovoltaic         5
TOT Generation       5
Renewables weight    5
Thermal              5
Wind                 5
Total Load           5
Load forecasted      5
Date                 0
Prices               0
dtype: int64

In [ ]:
# drop NaN rows
merged_data = merged_data.dropna().reset_index(drop=True)

In [33]:
merged_data['Date'].diff().value_counts().head()

Date
0 days 01:00:00    43803
0 days 02:00:00       10
Name: count, dtype: int64

Now we can export the merged dataset for the next phase, which is proper data analysis.

In [35]:
# export merged_data
merged_data.to_csv(clean_dataset_path/ 'merged-dataset.csv', index=False)

Back: [3.3. Energy load data cleaning](3.3_load-data-cleaning.ipynb)  

Next: [4 Exploratory Data Analysis]()